# Notebook 3 – Pipeline de Monitoramento e Avaliação

## Aula 5: Drift em Embeddings e Testes de Duas Amostras

### Objetivos
- Construir um **pipeline de monitoramento de drift em produção**
- Implementar **alertas baseados em limiares de MMD**
- Visualizar drift temporal com **projeções UMAP contínuas**
- Gerar **relatórios de drift** combinando múltiplos métodos
- Integrar detecção com **classificador adversário** no pipeline de MLOps

### Contexto Teórico (Documento da Aula 5)
> *"Monitorar continuamente o data drift é uma prática fundamental em pipelines
> de MLOps modernos."* (Microsoft, 2023)

> *"O DriftLens demonstrou ganhos de velocidade (até 5x mais rápido) e melhor
> acurácia na detecção de drift em dados de texto, imagens e voz."* (Greco et al., 2024)

**Vídeos relacionados:** Vídeo 4 (Pipeline de monitoramento de embeddings em produção)

In [ ]:
# Imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))

from src.utils import set_seed, generate_synthetic_dataset, load_model, save_metrics
from src.data_preprocessing import DataPreprocessor
from src.model import EmbeddingDriftDetector
from src.evaluation import (
    calculate_drift_metrics,
    plot_embedding_distributions,
    plot_drift_scores,
    plot_mmd_heatmap,
    generate_drift_report,
)
from src.training import compute_reference_statistics, cross_validate_detector

plt.style.use('seaborn-v0_8-whitegrid')
set_seed(42)

print('Imports carregados com sucesso!')

## 1. Carregamento dos Dados e Modelo Treinado

In [ ]:
# Carrega dados
dataset_path = Path('..') / 'data' / 'raw' / 'dataset.csv'
if not dataset_path.exists():
    generate_synthetic_dataset(output_path=str(dataset_path))

preprocessor = DataPreprocessor(embedding_dim=64)
df = preprocessor.load_data(str(dataset_path))
df = preprocessor.clean_data()

emb_cols = preprocessor.embedding_cols
emb_ref = df[df['period'] == 'reference'][emb_cols].values
emb_stable = df[df['period'] == 'production_stable'][emb_cols].values
emb_drift = df[df['period'] == 'production_drift'][emb_cols].values

# Tenta carregar o detector treinado no notebook anterior
try:
    detector = load_model('../outputs/models/mmd_detector.joblib')
    print('Detector MMD carregado de outputs/models/mmd_detector.joblib')
except FileNotFoundError:
    print('Detector não encontrado. Treinando novo...')
    detector = EmbeddingDriftDetector(method='mmd', gamma=0.1, n_permutations=100)
    detector.fit(emb_ref[:2000])

print(f'\nDados: {len(emb_ref)} ref, {len(emb_stable)} estável, {len(emb_drift)} drift')

## 2. Avaliação Completa com Múltiplos Métodos

Conforme a seção *Mercado, Cases e Tendências* da Aula 5:

> *"Combinar múltiplas técnicas e contextos de avaliação é a chave para um
> monitoramento efetivo de drift."* (Samuylova, 2023)

In [ ]:
# Avaliação com 3 métodos: dados estáveis (sem drift esperado)
print('=== Avaliação: Referência vs Produção Estável ===')
metrics_stable = calculate_drift_metrics(
    emb_ref[:1000], emb_stable[:1000],
    methods=['mmd', 'ks', 'adversarial'],
    gamma=0.1, n_permutations=100,
)

report_stable = generate_drift_report(metrics_stable)
print(report_stable)

In [ ]:
# Avaliação com 3 métodos: dados com drift
print('=== Avaliação: Referência vs Produção com Drift ===')
metrics_drift = calculate_drift_metrics(
    emb_ref[:1000], emb_drift[:1000],
    methods=['mmd', 'ks', 'adversarial'],
    gamma=0.1, n_permutations=100,
)

report_drift = generate_drift_report(metrics_drift)
print(report_drift)

# Salva métricas
save_metrics(metrics_stable, '../outputs/logs/metrics_stable.json')
save_metrics(metrics_drift, '../outputs/logs/metrics_drift.json')
print('Métricas salvas em outputs/logs/')

## 3. Visualização de Embeddings com UMAP/t-SNE

Conforme discutido no Vídeo 4 da Aula 5, a projeção UMAP permite a
**visualização contínua de drift** no pipeline de MLOps.

In [ ]:
# Visualização: Referência vs Produção Estável
labels_ref = df[df['period'] == 'reference']['category'].values[:1000]
labels_stable = df[df['period'] == 'production_stable']['category'].values[:1000]

fig = plot_embedding_distributions(
    emb_ref[:1000], emb_stable[:1000],
    method='tsne',
    labels_ref=labels_ref,
    labels_prod=labels_stable,
    title='Referência vs Produção Estável (Sem Drift Esperado)',
    save_path='../outputs/figures/embeddings_sem_drift.png',
)
plt.show()

In [ ]:
# Visualização: Referência vs Produção com Drift
labels_drift = df[df['period'] == 'production_drift']['category'].values[:1000]

fig = plot_embedding_distributions(
    emb_ref[:1000], emb_drift[:1000],
    method='tsne',
    labels_ref=labels_ref,
    labels_prod=labels_drift,
    title='Referência vs Produção com Drift (Drift Semântico Simulado)',
    save_path='../outputs/figures/embeddings_com_drift.png',
)
plt.show()

## 4. Heatmap do Kernel RBF

O heatmap do kernel mostra visualmente como o kernel RBF percebe a similaridade
entre amostras de referência e produção. Blocos diagonais claros indicam alta
similaridade intra-grupo; blocos off-diagonal escuros indicam drift.

In [ ]:
# Heatmap do kernel para cenário com drift
fig = plot_mmd_heatmap(
    emb_ref[:200], emb_drift[:200],
    gamma=0.1,
    title='Heatmap Kernel RBF: Referência vs Produção com Drift\n'
          '(Gretton et al., 2012 — k(u,v) = exp(-γ||u-v||²))',
    save_path='../outputs/figures/kernel_heatmap.png',
)
plt.show()

## 5. Pipeline de Monitoramento Temporal

Conforme o Vídeo 4 da Aula 5, um pipeline de monitoramento em produção deve:
1. Dividir dados de produção em **batches temporais**
2. Calcular scores de drift em cada batch
3. Comparar com **limiares calibrados** (do baseline)
4. Disparar **alertas** quando drift é detectado

Conforme Microsoft (2023), plataformas como Azure ML já incorporam monitores de drift
capazes de **disparar alertas preventivos**.

In [ ]:
# Simula monitoramento temporal com batches
# Pipeline: a cada batch de dados de produção, calcula MMD vs referência

# Calibra limiar
ref_stats = compute_reference_statistics(emb_ref[:2000], gamma=0.1, n_bootstrap=200)
threshold = ref_stats['threshold_95']
print(f'Limiar de alerta (95%): {threshold:.6f}')

# Simula batches temporais
# Primeiros batches: produção estável (sem drift)
# Últimos batches: produção com drift (drift crescente)
batch_size = 200
n_stable_batches = len(emb_stable) // batch_size
n_drift_batches = len(emb_drift) // batch_size

scores_timeline = []
batch_labels = []
alerts = []

detector_mon = EmbeddingDriftDetector(method='mmd', gamma=0.1, n_permutations=50)
detector_mon.fit(emb_ref[:2000])

# Batches estáveis
for i in range(n_stable_batches):
    batch = emb_stable[i * batch_size : (i + 1) * batch_size]
    score = detector_mon.score(batch)
    scores_timeline.append(score)
    batch_labels.append(f'S{i+1}')
    alerts.append(score > threshold)

# Batches com drift
for i in range(n_drift_batches):
    batch = emb_drift[i * batch_size : (i + 1) * batch_size]
    score = detector_mon.score(batch)
    scores_timeline.append(score)
    batch_labels.append(f'D{i+1}')
    alerts.append(score > threshold)

print(f'\nTotal de batches: {len(scores_timeline)}')
print(f'Alertas disparados: {sum(alerts)} / {len(alerts)}')

In [ ]:
# Visualização do monitoramento temporal
fig = plot_drift_scores(
    scores_timeline,
    timestamps=batch_labels,
    threshold=threshold,
    title='Monitoramento Temporal de Drift (MMD²)\n'
          'S=Estável, D=Drift — Pipeline conforme Vídeo 4 da Aula 5',
    ylabel='MMD²',
    save_path='../outputs/figures/monitoramento_temporal.png',
)
plt.show()

print('Observe como os scores de drift permanecem baixos nos batches estáveis (S)')
print('e saltam acima do limiar nos batches com drift (D), disparando alertas.')
print('\nIsto implementa o conceito de "alertas baseados em limiares de MMD"')
print('discutido no Vídeo 4 da Aula 5.')

## 6. Cross-Validação do Detector

Avalia a consistência da detecção de drift usando múltiplos subconjuntos.

In [ ]:
# Cross-validação da detecção de drift
cv_result = cross_validate_detector(
    detector_mon, emb_ref[:2000], emb_drift, n_splits=5, seed=42
)

print('=== Cross-Validação do Detector de Drift ===')
print(f'Score médio:      {cv_result["mean_score"]:.6f} ± {cv_result["std_score"]:.6f}')
print(f'Consistência:     {cv_result["consistency"]:.0%}')
print(f'\nDetecção por fold: {cv_result["drift_detected_per_fold"]}')
print(f'Scores por fold:   {[f"{s:.6f}" for s in cv_result["scores"]]}')

if cv_result['consistency'] >= 0.8:
    print('\n✓ Alta consistência: drift detectado em ≥80% dos folds')
else:
    print('\n⚠ Baixa consistência: drift não é consistentemente detectado')

## 7. Extração de Embeddings com BERT (Demonstração)

Conforme o Snippet 1 do Hands On da Aula 5:

> *"Para detectar drifts em dados não estruturados, primeiro precisamos representá-los
> numericamente [...] usando um modelo pré-treinado de linguagem (BERT)."*
> (Devlin et al., 2019)

**Nota:** Esta célula requer `transformers` e `torch` instalados.

In [ ]:
# Demonstração de extração de embeddings com BERT
# Implementa Snippet 1 da seção Hands On do documento da Aula 5
try:
    from transformers import BertTokenizer, BertModel
    import torch

    # Carrega modelo pré-treinado (Devlin et al., 2019)
    print('Carregando BERT-base-uncased...')
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    bert_model = BertModel.from_pretrained('bert-base-uncased')

    # Textos de exemplo do caso TrendCast
    ref_texts = [
        'The user posted about a local sports match.',
        'The weather today is sunny with clear skies.',
        'New movie release gets great audience reviews.',
    ]
    new_texts = [
        'The election results shocked the nation.',
        'Political debates are trending on social media.',
        'Government policy changes affect market prices.',
    ]

    def get_bert_embeddings(texts):
        """Extrai embeddings [CLS] do BERT (Devlin et al., 2019)."""
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
        with torch.no_grad():
            outputs = bert_model(**inputs)
        return outputs.last_hidden_state[:, 0, :].numpy()

    emb_bert_ref = get_bert_embeddings(ref_texts)
    emb_bert_new = get_bert_embeddings(new_texts)

    print(f'Embedding referência: {emb_bert_ref.shape}')
    print(f'Embedding novos:      {emb_bert_new.shape}')

    # MMD entre embeddings BERT
    det = EmbeddingDriftDetector(method='mmd', gamma=0.001)
    mmd_bert = det.compute_mmd(emb_bert_ref, emb_bert_new, gamma=0.001)
    print(f'\nMMD² (BERT embeddings): {mmd_bert:.6f}')
    print('\nNota: com poucos textos, o MMD pode não ser estatisticamente significativo.')
    print('Em produção, use centenas/milhares de amostras por batch.')

except ImportError:
    print('transformers e/ou torch não instalados.')
    print('Para usar BERT, instale: pip install transformers torch')
    print('\nOs resultados com dataset sintético demonstram os mesmos conceitos.')

## 8. Relatório Final e Dashboard

Conforme discutido na seção *Mercado, Cases e Tendências*:

> *"Cursos e treinamentos de MLOps agora enfatizam a necessidade de se planejar com
> antecedência para drifts inevitáveis, recomendando práticas como validação contínua,
> detection dashboards e pipelines de re-treinamento automatizados."*

In [ ]:
# Dashboard resumo de monitoramento
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Timeline de drift scores
ax = axes[0, 0]
ax.plot(scores_timeline, 'o-', color='steelblue', markersize=4)
ax.axhline(threshold, color='red', linestyle='--', label=f'Limiar ({threshold:.5f})')
ax.axvline(n_stable_batches - 0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Batch')
ax.set_ylabel('MMD²')
ax.set_title('Timeline de Drift Scores')
ax.legend(fontsize=8)
ax.text(n_stable_batches / 2, max(scores_timeline) * 0.9, 'Estável',
        ha='center', fontsize=9, color='green')
ax.text(n_stable_batches + n_drift_batches / 2, max(scores_timeline) * 0.9, 'Drift',
        ha='center', fontsize=9, color='red')

# 2. Comparação dos métodos
ax = axes[0, 1]
methods = ['MMD', 'KS', 'Adversarial']
drift_detected = [
    metrics_drift.get('mmd', {}).get('drift_detected', False),
    metrics_drift.get('ks', {}).get('drift_detected', False),
    metrics_drift.get('adversarial', {}).get('drift_detected', False),
]
colors = ['coral' if d else 'green' for d in drift_detected]
ax.bar(methods, [1 if d else 0 for d in drift_detected], color=colors)
ax.set_ylabel('Drift Detectado')
ax.set_title('Consenso dos Métodos (Produção com Drift)')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Não', 'Sim'])

# 3. Distribuição de normas
ax = axes[1, 0]
norms_r = np.linalg.norm(emb_ref, axis=1)
norms_d = np.linalg.norm(emb_drift, axis=1)
ax.hist(norms_r, bins=40, alpha=0.5, label='Referência', density=True, color='steelblue')
ax.hist(norms_d, bins=40, alpha=0.5, label='Drift', density=True, color='coral')
ax.set_xlabel('Norma L2')
ax.set_ylabel('Densidade')
ax.set_title('Distribuição de Normas dos Embeddings')
ax.legend(fontsize=9)

# 4. Alertas acumulados
ax = axes[1, 1]
cumulative_alerts = np.cumsum(alerts)
ax.plot(cumulative_alerts, 'r-', linewidth=2)
ax.axvline(n_stable_batches - 0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Batch')
ax.set_ylabel('Alertas Acumulados')
ax.set_title('Alertas de Drift Acumulados')
ax.fill_between(range(len(cumulative_alerts)), cumulative_alerts, alpha=0.2, color='red')

fig.suptitle('Dashboard de Monitoramento de Drift em Embeddings\n'
             '(Pipeline de MLOps conforme Vídeo 4 da Aula 5)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/dashboard_monitoramento.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Salva relatório final
report_final = generate_drift_report(
    metrics_drift,
    output_path='../outputs/logs/drift_report.txt',
)
print(report_final)
print('\nRelatório salvo em outputs/logs/drift_report.txt')

## Resumo

Neste notebook, construímos um **pipeline completo de monitoramento de drift em embeddings**:

1. **Avaliação multi-método**: MMD, KS e classificador adversário concordam na detecção
2. **Visualização UMAP/t-SNE**: separação visual clara entre dados de referência e drift
3. **Monitoramento temporal**: scores de drift por batch com limiares de alerta
4. **Dashboard**: visão unificada do status de drift no pipeline de MLOps
5. **Extração BERT**: demonstração de como extrair embeddings reais de textos

### Conclusões (conforme seção 'O que você viu nesta aula')

- O **drift semântico** é mais difícil de detectar que o superficial, mas técnicas como
  MMD e classificadores adversários são eficazes quando aplicadas em espaço de embeddings
- **Monitorar continuamente** os dados de produção é essencial para manter a qualidade
  dos modelos de ML (Microsoft, 2023; Sculley et al., 2015)
- **Combinar múltiplas técnicas** (MMD + KS + adversarial) aumenta a confiança na detecção

### Referências Principais
- Gretton et al. (2012) — MMD
- Massey (1951) — KS
- Lopez-Paz & Oquab (2017) — Classificador adversário
- Devlin et al. (2019) — BERT
- Greco et al. (2024) — DriftLens
- Feldhans et al. (2021) — Drift em embeddings de texto